# PyVPP — Test de descarga Sentinel-2 sobre Doñana

Notebook mínimo para validar el flujo `CDSEDownload` end-to-end:
search → download → mosaic per-band → clip al AOI.

Genera un `mosaic_<BAND>_rec.tif` por cada banda solicitada, en su
resolución nativa (10 m o 20 m).

**Requisitos previos:**

1. Credenciales CDSE en `~/.pyvpp/config.toml` o variables de entorno
   `CDSE_USER` / `CDSE_PASSWORD`. Regístrate en https://dataspace.copernicus.eu/.
2. Entorno con `rasterio`, `geopandas`, `shapely`, `pyproj`, `requests`, `fiona`.

## 1. Configuración

Edita `SHAPE` con la ruta al shapefile del AOI. El resto de variables
tienen valores por defecto razonables para un test corto sobre Doñana.

In [1]:
# === EDITAR ESTO ============================================================
SHAPE = '/media/diego/31F8C0B3792FC3B6/EBD/cartografia_up/PN_Doñana/Limites/Limite_Doñana.shp'   # ← pon aquí el path a tu shapefile
# ============================================================================

# Una ventana corta. Con dos semanas hay 1–2 pasadas S2 sobre Doñana.
DATES = ('2023-06-09', '2023-06-11')

# Bandas del protocolo Doñana: 4 a 10 m + red edge a 20 m + 2 SWIR a 20 m + SCL.
BANDS = ['B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08', 'B8A', 'B11', 'B12', 'SCL']

# Forzar UTM 29 (huso de Doñana). PyVPP descartará tiles de otros husos.
UTM_ZONE = 29

OUTDIR = './pyvpp_donana_test'

# Nivel de procesado: L2A (BOA, atmosféricamente corregido). SCL solo está en L2A.
PROCESSING_LEVEL = 'L2A'

## 2. Búsqueda (sin descargar)

Antes de tirar GBs, listar lo que CDSE devuelve para esa AOI/fecha y verificar
que el filtro UTM y la cobertura espacial son los esperados.

In [2]:
from pyvpp.cdse import CDSEDownload

dl = CDSEDownload(
    shape=SHAPE,
    dates=DATES,
    bands=BANDS,
    outdir=OUTDIR,
    utm_zone=UTM_ZONE,
    processing_level=PROCESSING_LEVEL,
)

products = dl.search()
print(f'\n{len(products)} productos tras filtro UTM {UTM_ZONE}:')
for p in products:
    size_gb = p.get('ContentLength', 0) / 1e9
    print(f"  - {p['Name']}  [{size_gb:.2f} GB]")

total_gb = sum(p.get('ContentLength', 0) for p in products) / 1e9
print(f'\nTotal a descargar: ~{total_gb:.1f} GB')

Searching for Sentinel-2 L2A products...
Filtered products by UTM zone 29: 4 → 2

2 productos tras filtro UTM 29:
  - S2A_MSIL2A_20230610T110621_N0510_R137_T29SQA_20240925T175900.SAFE  [1.14 GB]
  - S2A_MSIL2A_20230610T110621_N0510_R137_T29SQB_20240925T175900.SAFE  [1.17 GB]

Total a descargar: ~2.3 GB


## 3. Descarga + mosaico + recorte

`run()` ejecuta todo: descarga los `.SAFE`, los descomprime, agrupa los JP2
por banda, mosaica los tiles que cubren el AOI y recorta al cutline.

**Tarda lo que tarde la red de CDSE** (típicamente 5–20 min para una pasada
completa sobre Doñana, según hora del día).

In [3]:
import time
t0 = time.time()
dl.run()
print(f'\nFinalizado en {(time.time() - t0) / 60:.1f} min')

Searching for Sentinel-2 L2A products...
Filtered products by UTM zone 29: 4 → 2
  [1/2] Descargando S2A_MSIL2A_20230610T110621_N0510_R137_T29SQB_20240925T175900.SAFE ...
  [2/2] Descargando S2A_MSIL2A_20230610T110621_N0510_R137_T29SQA_20240925T175900.SAFE ...
  20230610:
    B02 (10m): 2 JP2 file(s)
    B03 (10m): 2 JP2 file(s)
    B04 (10m): 2 JP2 file(s)
    B05 (20m): 2 JP2 file(s)
    B06 (20m): 2 JP2 file(s)
    B07 (20m): 2 JP2 file(s)
    B08 (10m): 2 JP2 file(s)
    B8A (20m): 2 JP2 file(s)
    B11 (20m): 2 JP2 file(s)
    B12 (20m): 2 JP2 file(s)
    SCL (20m): 2 JP2 file(s)

Mosaicando fecha 20230610 ...
✅ B02: ./pyvpp_donana_test/20230610_s2msi/20230610_s2msi_b02_blue.tif
✅ B03: ./pyvpp_donana_test/20230610_s2msi/20230610_s2msi_b03_green.tif
✅ B04: ./pyvpp_donana_test/20230610_s2msi/20230610_s2msi_b04_red.tif
✅ B05: ./pyvpp_donana_test/20230610_s2msi/20230610_s2msi_b05_re1.tif
✅ B06: ./pyvpp_donana_test/20230610_s2msi/20230610_s2msi_b06_re2.tif
✅ B07: ./pyvpp_donana_test/20

## 4. Inspección de la salida

Listar los GeoTIFF generados con sus dimensiones, resolución y dtype para
comprobar que cada banda quedó en su resolución nativa y que SCL conservó su
tipo categórico (uint8).

In [ ]:
import os, glob
import rasterio
import numpy as np

scl_meanings = {
    0: "nodata", 1: "saturated", 2: "dark", 3: "shadow",
    4: "vegetation", 5: "bare", 6: "water",
    7: "unclassified", 8: "cloud_med", 9: "cloud_high",
    10: "cirrus", 11: "snow",
}

for path in sorted(glob.glob(os.path.join(OUTDIR, "*_s2msi", "*.tif"))):
    with rasterio.open(path) as src:
        size_mb = os.path.getsize(path) / 1e6
        fname = os.path.basename(path)
        print(f"{fname}: {src.width}x{src.height} @ {src.res[0]:.0f}m, "
              f"{src.dtypes[0]}, {size_mb:.1f} MB")
        if "scl_fmask" in fname:
            arr = src.read(1)
            uniq, cnt = np.unique(arr, return_counts=True)
            print("  SCL classes:")
            for u, c in zip(uniq.tolist(), cnt.tolist()):
                pct = 100 * c / arr.size
                label = scl_meanings.get(u, "?")
                print(f"    {u:2d} ({label:>14s}): {c:>10d} px  ({pct:5.2f}%)")
